# Inviscid Burgers equation before breaking: PyCUDA MacCormack solver

This notebook is the GPU counterpart of the pre-breaking NumPy solver for

$$
u_t+\left(\frac{u^2}{2}\right)_x=0,
\qquad 0\leq x\leq 2,
\qquad 0\leq t<t_b=1.
$$

The conservative MacCormack update is split into a forward predictor and a backward corrector.  The implementation uses direct, coalesced global-memory accesses and three rolling device arrays.  It also contains an independent NumPy implementation and an MP4 animation comparing GPU, CPU, and the exact characteristic solution.  Every public driver restricts the computation to times strictly below the breaking time.


## 1. Colab setup


In [1]:
!pip install -q pycuda


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.5 MB/s eta 0:00:00


## 2. Imports and CUDA interface

PyCUDA compiles the CUDA kernels at run time and manages device allocation and kernel launches.  NumPy supplies the independent CPU reference, the exact pre-breaking field, and the validation diagnostics.


In [2]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path

import numpy as np

try:
    import pycuda.autoinit  # noqa: F401
    import pycuda.driver as cuda
    import pycuda.gpuarray as gpuarray
    from pycuda.compiler import SourceModule
except ImportError as exc:
    raise SystemExit(
        "PyCUDA is required. Install a CUDA-compatible PyCUDA build and run "
        "the notebook on an NVIDIA GPU."
    ) from exc

Array = np.ndarray
BREAKING_TIME = 1.0
LEFT_VALUE = 1.0
RIGHT_VALUE = 0.0


## 3. CUDA kernels

Two kernels implement one conservative MacCormack step.  The predictor evaluates

$$
\widetilde U_i=U_i^n-\lambda\left[f(U_{i+1}^n)-f(U_i^n)\right],
\qquad \lambda=\frac{\Delta t}{\Delta x},
$$

for $i=0,\ldots,N-1$ and assigns the known right-boundary trace at $i=N$.  The corrector evaluates the opposite flux difference for $i=1,\ldots,N-1$ and assigns both endpoint values.  Neighboring threads read neighboring array entries, so the principal global-memory accesses are coalesced.  Separate kernels are necessary because every corrector thread must see the completed predictor field.


In [3]:
CUDA_SOURCE = r"""
extern "C" {

__device__ __forceinline__ double flux(const double u)
{
    return 0.5 * u * u;
}

__global__ void predictor(
    const double *__restrict__ u,
    double *__restrict__ u_star,
    const double lambda,
    const int npoints,
    const double right_value)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= npoints) return;

    if (i < npoints - 1) {
        u_star[i] = u[i]
                  - lambda * (flux(u[i + 1]) - flux(u[i]));
    } else {
        u_star[i] = right_value;
    }
}

__global__ void corrector(
    const double *__restrict__ u,
    const double *__restrict__ u_star,
    double *__restrict__ u_next,
    const double lambda,
    const int npoints,
    const double left_value,
    const double right_value)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= npoints) return;

    if (i == 0) {
        u_next[i] = left_value;
    } else if (i == npoints - 1) {
        u_next[i] = right_value;
    } else {
        u_next[i] = 0.5 * (
            u[i] + u_star[i]
            - lambda * (flux(u_star[i]) - flux(u_star[i - 1]))
        );
    }
}

} // extern "C"
"""


## 4. Initial data and exact pre-breaking characteristic reference


In [4]:
def initial_condition(x: Array) -> Array:
    """Return u(x,0)=1-x on [0,1] and zero on (1,2]."""
    x = np.asarray(x, dtype=np.float64)
    return np.where(x <= 1.0, 1.0 - x, 0.0)


def exact_prebreaking_solution(x: Array, time: float) -> Array:
    """Evaluate the smooth characteristic solution for 0 <= time < 1."""
    if not (0.0 <= time < BREAKING_TIME):
        raise ValueError("time must satisfy 0 <= time < 1")

    x = np.asarray(x, dtype=np.float64)
    exact = np.zeros_like(x)
    exact[x < time] = LEFT_VALUE
    smooth = (x >= time) & (x <= 1.0)
    exact[smooth] = (1.0 - x[smooth]) / (1.0 - time)
    exact[x > 1.0] = RIGHT_VALUE
    exact[0] = LEFT_VALUE
    exact[-1] = RIGHT_VALUE
    return exact


## 5. CPU reference: the same conservative MacCormack method

The NumPy routines reproduce exactly the predictor, corrector, and problem-specific endpoint treatment used by the GPU kernels.  They remain independent from the GPU code so that CPU--GPU agreement validates the CUDA implementation rather than two wrappers around the same routine.


In [5]:
def maccormack_step_cpu(u: Array, lam: float) -> Array:
    """Advance one conservative MacCormack step with fixed endpoint data."""
    u = np.asarray(u, dtype=np.float64)
    flux = 0.5 * u * u

    u_star = np.empty_like(u)
    u_star[:-1] = u[:-1] - lam * (flux[1:] - flux[:-1])
    u_star[-1] = RIGHT_VALUE

    flux_star = 0.5 * u_star * u_star
    u_next = np.empty_like(u)
    u_next[0] = LEFT_VALUE
    u_next[1:-1] = 0.5 * (
        u[1:-1] + u_star[1:-1]
        - lam * (flux_star[1:-1] - flux_star[:-2])
    )
    u_next[-1] = RIGHT_VALUE
    return u_next


def solve_cpu_reference(initial: Array, *, lam: float, nsteps: int) -> Array:
    """Return the NumPy MacCormack solution after ``nsteps`` steps."""
    if nsteps < 0:
        raise ValueError("nsteps must be non-negative")
    u = np.asarray(initial, dtype=np.float64).copy()
    for _ in range(nsteps):
        u = maccormack_step_cpu(u, lam)
    return u


## 6. Kernel compilation and GPU solver


In [6]:
class BurgersMacCormackKernels:
    """Compile the CUDA source once and expose predictor and corrector."""

    def __init__(self) -> None:
        module = SourceModule(CUDA_SOURCE, options=["-O3"])
        self.predictor = module.get_function("predictor")
        self.corrector = module.get_function("corrector")


def _launch_geometry(npoints: int, block_size: int):
    if block_size <= 0:
        raise ValueError("block_size must be positive")
    device = cuda.Context.get_device()
    maximum = device.get_attribute(cuda.device_attribute.MAX_THREADS_PER_BLOCK)
    if block_size > maximum:
        raise ValueError(
            f"block_size={block_size} exceeds the device limit {maximum}"
        )
    return (block_size, 1, 1), ((npoints + block_size - 1) // block_size, 1, 1)


def solve_gpu(
    initial: Array,
    *,
    lam: float,
    nsteps: int,
    block_size: int = 256,
    kernels: BurgersMacCormackKernels | None = None,
) -> tuple[Array, float]:
    """Run conservative MacCormack on the GPU and return field and kernel time."""
    initial = np.asarray(initial, dtype=np.float64)
    if initial.ndim != 1:
        raise ValueError("initial must be a one-dimensional array")
    if initial.size < 3:
        raise ValueError("at least 3 grid points are required")
    if not (0.0 < lam <= 1.0):
        raise ValueError("lam must satisfy 0 < lam <= 1")
    if nsteps < 0:
        raise ValueError("nsteps must be non-negative")
    if nsteps == 0:
        return initial.copy(), 0.0

    kernels = kernels or BurgersMacCormackKernels()
    npoints_host = int(initial.size)
    block, grid = _launch_geometry(npoints_host, block_size)

    u = gpuarray.to_gpu(initial)
    u_star = gpuarray.empty_like(u)
    u_next = gpuarray.empty_like(u)

    start = cuda.Event()
    stop = cuda.Event()
    start.record()

    for _ in range(nsteps):
        kernels.predictor(
            u, u_star, np.float64(lam), np.int32(npoints_host),
            np.float64(RIGHT_VALUE), block=block, grid=grid,
        )
        kernels.corrector(
            u, u_star, u_next, np.float64(lam), np.int32(npoints_host),
            np.float64(LEFT_VALUE), np.float64(RIGHT_VALUE),
            block=block, grid=grid,
        )
        u, u_next = u_next, u

    stop.record()
    stop.synchronize()
    gpu_seconds = 1.0e-3 * start.time_till(stop)
    return u.get(), gpu_seconds


## 7. Problem setup and single-run validation


In [7]:
@dataclass(frozen=True)
class RunResult:
    x: Array
    gpu: Array
    cpu: Array
    exact: Array
    dx: float
    dt: float
    nsteps: int
    courant: float
    gpu_seconds: float
    gpu_relative_l2: float
    cpu_relative_l2: float
    gpu_cpu_difference: float


def problem_setup(
    *, nx: int, cfl: float, final_time: float, length: float
):
    """Build the grid and a fixed step that reaches the final time exactly."""
    if nx < 2:
        raise ValueError("nx must be at least 2")
    if length <= 1.0:
        raise ValueError("length must exceed 1 so the initial support fits")
    if not (0.0 < final_time < BREAKING_TIME):
        raise ValueError("final_time must satisfy 0 < final_time < 1")
    if not (0.0 < cfl <= 1.0):
        raise ValueError("cfl must satisfy 0 < cfl <= 1")

    dx = length / nx
    x = np.linspace(0.0, length, nx + 1, dtype=np.float64)
    initial = initial_condition(x)
    tentative_dt = cfl * dx  # max |u| = 1 for this pre-breaking test
    nsteps = max(1, int(np.ceil(final_time / tentative_dt)))
    dt = final_time / nsteps
    courant = dt / dx
    return x, initial, dx, dt, courant, nsteps


def run_validation(
    *,
    nx: int = 4096,
    cfl: float = 0.4,
    final_time: float = 0.90,
    length: float = 2.0,
    block_size: int = 256,
) -> RunResult:
    """Compare GPU and CPU MacCormack fields with the smooth exact solution."""
    x, initial, dx, dt, courant, nsteps = problem_setup(
        nx=nx, cfl=cfl, final_time=final_time, length=length
    )

    kernels = BurgersMacCormackKernels()
    gpu, gpu_seconds = solve_gpu(
        initial, lam=courant, nsteps=nsteps,
        block_size=block_size, kernels=kernels,
    )
    cpu = solve_cpu_reference(initial, lam=courant, nsteps=nsteps)
    exact = exact_prebreaking_solution(x, final_time)

    exact_norm = max(float(np.linalg.norm(exact)), 1.0e-30)
    cpu_norm = max(float(np.linalg.norm(cpu)), 1.0e-30)
    result = RunResult(
        x=x, gpu=gpu, cpu=cpu, exact=exact, dx=dx, dt=dt,
        nsteps=nsteps, courant=courant, gpu_seconds=gpu_seconds,
        gpu_relative_l2=float(np.linalg.norm(gpu - exact) / exact_norm),
        cpu_relative_l2=float(np.linalg.norm(cpu - exact) / exact_norm),
        gpu_cpu_difference=float(np.linalg.norm(gpu - cpu) / cpu_norm),
    )

    print("=== Conservative MacCormack validation: pre-breaking regime ===")
    print(f"grid cells                = {nx}")
    print(f"time steps                = {result.nsteps}")
    print(f"final time                = {final_time:.12e}")
    print(f"dt                        = {result.dt:.12e}")
    print(f"effective CFL number      = {result.courant:.12e}")
    print(f"GPU kernel time           = {result.gpu_seconds:.6e} s")
    print(f"GPU relative L2 error     = {result.gpu_relative_l2:.12e}")
    print(f"CPU relative L2 error     = {result.cpu_relative_l2:.12e}")
    print(f"GPU-CPU difference        = {result.gpu_cpu_difference:.12e}")
    return result


In [8]:
# Representative pre-breaking numerical validation.
validation = run_validation(
    nx=4096,
    cfl=0.4,
    final_time=0.90,
    block_size=256,
)


=== Conservative MacCormack validation: pre-breaking regime ===
grid cells                = 4096
time steps                = 4608
final time                = 9.000000000000e-01
dt                        = 1.953125000000e-04
effective CFL number      = 4.000000000000e-01
GPU kernel time           = 2.169842e-01 s
GPU relative L2 error     = 3.076138112984e-04
CPU relative L2 error     = 3.076138112983e-04
GPU-CPU difference        = 1.540417645778e-14


## 8. GPU--CPU--exact validation animation

The animation is a validation experiment rather than a performance benchmark.  Device-to-host copies occur only at selected time levels.  Complete fields and error norms use all $N_x+1$ samples, whereas only the curves sent to Matplotlib may be spatially decimated.  The requested final time is checked against $t_b=1$ before either solver is run.


In [9]:
def solve_cpu_with_snapshots(
    initial: Array, *, lam: float, nsteps: int, snapshot_steps: Array
) -> Array:
    """Store selected NumPy MacCormack time levels."""
    requested = {int(step) for step in snapshot_steps}
    history = {}
    u = np.asarray(initial, dtype=np.float64).copy()
    if 0 in requested:
        history[0] = u.copy()
    for step in range(1, nsteps + 1):
        u = maccormack_step_cpu(u, lam)
        if step in requested:
            history[step] = u.copy()
    return np.stack([history[int(step)] for step in snapshot_steps])


def solve_gpu_with_snapshots(
    initial: Array,
    *,
    lam: float,
    nsteps: int,
    snapshot_steps: Array,
    block_size: int = 256,
    kernels: BurgersMacCormackKernels | None = None,
) -> Array:
    """Store selected GPU MacCormack time levels."""
    requested = {int(step) for step in snapshot_steps}
    history = {}
    initial = np.asarray(initial, dtype=np.float64)
    if 0 in requested:
        history[0] = initial.copy()

    kernels = kernels or BurgersMacCormackKernels()
    npoints_host = int(initial.size)
    block, grid = _launch_geometry(npoints_host, block_size)
    u = gpuarray.to_gpu(initial)
    u_star = gpuarray.empty_like(u)
    u_next = gpuarray.empty_like(u)

    for step in range(1, nsteps + 1):
        kernels.predictor(
            u, u_star, np.float64(lam), np.int32(npoints_host),
            np.float64(RIGHT_VALUE), block=block, grid=grid,
        )
        kernels.corrector(
            u, u_star, u_next, np.float64(lam), np.int32(npoints_host),
            np.float64(LEFT_VALUE), np.float64(RIGHT_VALUE),
            block=block, grid=grid,
        )
        u, u_next = u_next, u
        if step in requested:
            history[step] = u.get()

    return np.stack([history[int(step)] for step in snapshot_steps])


def create_gpu_cpu_reference_animation(
    *,
    nx: int = 4096,
    cfl: float = 0.4,
    final_time: float = 0.90,
    length: float = 2.0,
    block_size: int = 256,
    max_frames: int = 120,
    max_plot_points: int = 4096,
    fps: int = 25,
    dpi: int = 120,
    output: str | Path = "burgers_maccormack_gpu_cpu_exact_4096.mp4",
    preview: bool = True,
):
    """Create a pre-breaking MP4 comparison of GPU, CPU, and exact fields."""
    if max_frames < 2:
        raise ValueError("max_frames must be at least 2")
    if max_plot_points < 2:
        raise ValueError("max_plot_points must be at least 2")

    import matplotlib.pyplot as plt
    from matplotlib.animation import FFMpegWriter, FuncAnimation

    x, initial, _, dt, courant, nsteps = problem_setup(
        nx=nx, cfl=cfl, final_time=final_time, length=length
    )
    nframes = min(max_frames, nsteps + 1)
    snapshot_steps = np.unique(
        np.rint(np.linspace(0, nsteps, nframes)).astype(int)
    )
    snapshot_times = snapshot_steps * dt

    kernels = BurgersMacCormackKernels()
    gpu_history = solve_gpu_with_snapshots(
        initial, lam=courant, nsteps=nsteps,
        snapshot_steps=snapshot_steps, block_size=block_size, kernels=kernels,
    )
    cpu_history = solve_cpu_with_snapshots(
        initial, lam=courant, nsteps=nsteps, snapshot_steps=snapshot_steps
    )

    final_denom = max(float(np.linalg.norm(cpu_history[-1])), 1.0e-30)
    final_gpu_cpu = float(
        np.linalg.norm(gpu_history[-1] - cpu_history[-1]) / final_denom
    )
    print(f"GPU-CPU relative difference at final time = {final_gpu_cpu:.12e}")

    plot_stride = max(1, int(np.ceil(x.size / max_plot_points)))
    plot_indices = np.arange(0, x.size, plot_stride, dtype=int)
    if plot_indices[-1] != x.size - 1:
        plot_indices = np.append(plot_indices, x.size - 1)
    x_plot = x[plot_indices]

    ymin = min(0.0, float(np.min(gpu_history)), float(np.min(cpu_history)))
    ymax = max(1.0, float(np.max(gpu_history)), float(np.max(cpu_history)))
    pad = max(0.08 * (ymax - ymin), 0.05)

    fig, ax = plt.subplots(figsize=(8.2, 4.9))
    gpu_line, = ax.plot([], [], linewidth=2.0, label="GPU PyCUDA")
    cpu_line, = ax.plot([], [], linewidth=1.7, label="CPU NumPy")
    exact_line, = ax.plot([], [], "--", linewidth=1.8, label="Exact")
    time_text = ax.text(0.02, 0.96, "", transform=ax.transAxes, va="top")
    error_text = ax.text(0.02, 0.86, "", transform=ax.transAxes, va="top")
    ax.set_xlim(float(x[0]), float(x[-1]))
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.set_title(
        f"Burgers MacCormack before breaking: GPU vs CPU vs exact; "
        f"Nx={nx}, C={courant:.3f}"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    fig.tight_layout()
    def initialize():
        for line in (gpu_line, cpu_line, exact_line):
            line.set_data([], [])
        time_text.set_text("")
        error_text.set_text("")
        return gpu_line, cpu_line, exact_line, time_text, error_text

    def update(frame):
        time_now = float(snapshot_times[frame])
        exact_now = exact_prebreaking_solution(x, time_now)
        gpu_now = gpu_history[frame]
        cpu_now = cpu_history[frame]
        gpu_line.set_data(x_plot, gpu_now[plot_indices])
        cpu_line.set_data(x_plot, cpu_now[plot_indices])
        exact_line.set_data(x_plot, exact_now[plot_indices])
        denominator = max(float(np.linalg.norm(exact_now)), 1.0e-30)
        gpu_error = float(np.linalg.norm(gpu_now - exact_now) / denominator)
        cpu_error = float(np.linalg.norm(cpu_now - exact_now) / denominator)
        gpu_cpu = float(
            np.linalg.norm(gpu_now - cpu_now)
            / max(float(np.linalg.norm(cpu_now)), 1.0e-30)
        )
        time_text.set_text(f"t = {time_now:.5f} < t_b")
        error_text.set_text(
            f"GPU error = {gpu_error:.3e}\n"
            f"CPU error = {cpu_error:.3e}\n"
            f"GPU-CPU = {gpu_cpu:.3e}"
        )
        return gpu_line, cpu_line, exact_line, time_text, error_text

    animation = FuncAnimation(
        fig, update, frames=len(snapshot_steps), init_func=initialize,
        interval=1000.0 / fps, blit=True,
    )
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    if not FFMpegWriter.isAvailable():
        plt.close(fig)
        raise RuntimeError(
            "FFmpeg is not available. In Google Colab it is normally "
            "preinstalled; otherwise install ffmpeg before creating the MP4."
        )
    writer = FFMpegWriter(
        fps=fps, bitrate=2200,
        metadata={"title": "Burgers MacCormack before breaking: GPU, CPU, exact"},
    )
    animation.save(str(output), writer=writer, dpi=dpi)
    plt.close(fig)
    print(f"Video saved to {output.resolve()} ({len(snapshot_steps)} frames).")
    if preview:
        from IPython.display import Video, display
        display(Video(str(output), embed=True))
    return output.resolve()


In [10]:
# GPU--CPU--exact animation, entirely before t_b=1.
video_path = create_gpu_cpu_reference_animation(
    nx=4096,
    cfl=0.4,
    final_time=0.90,
    max_frames=120,
    max_plot_points=4096,
    output="burgers_maccormack_gpu_cpu_exact_4096.mp4",
    preview=True,
)


GPU-CPU relative difference at final time = 1.540417645778e-14
Video saved to /content/burgers_maccormack_gpu_cpu_exact_4096.mp4 (120 frames).


## 9. Optional command-line interface

The notebook does not call `main()` automatically, so it is safe to run from top to bottom in Colab.  If exported to a Python script, call `main()` from the usual `if __name__ == "__main__"` block.  The parser and the setup routine both reject a final time at or beyond the breaking time.


In [11]:
def build_parser() -> argparse.ArgumentParser:
    """Build the command-line parser without reading process arguments."""
    parser = argparse.ArgumentParser(
        description="Pre-breaking PyCUDA MacCormack solver for 1D Burgers."
    )
    parser.add_argument("--nx", type=int, default=4096)
    parser.add_argument("--cfl", type=float, default=0.4)
    parser.add_argument("--final-time", type=float, default=0.90)
    parser.add_argument("--length", type=float, default=2.0)
    parser.add_argument("--block-size", type=int, default=256)
    return parser


def main(argv=None) -> int:
    """Command-line entry point; pass an explicit argv list in notebooks."""
    args = build_parser().parse_args(argv)
    run_validation(
        nx=args.nx,
        cfl=args.cfl,
        final_time=args.final_time,
        length=args.length,
        block_size=args.block_size,
    )
    return 0
